In [1]:
import pandas as pd
from PIL import ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True   # tolerate truncated files
df = pd.read_csv("/kaggle/input/datasets/abhishekbuddiga06/embryo-dataset/embryo_dataset_annotations/embryo_dataset_annotations/AA83-7_phases.csv")
df.head()

,tPB2,5,24
0,tPNa,25,88
1,tPNf,89,97
2,t2,98,171
3,t3,172,177
4,t4,178,191


In [2]:
import os
import pandas as pd

FOLDER_PATH = "/kaggle/input/datasets/abhishekbuddiga06/embryo-dataset/embryo_dataset_annotations"  # change this

unique_stages = set()

# Loop through all CSV files
for root, dirs, files in os.walk(FOLDER_PATH):
    for file in files:
        if file.endswith(".csv"):
            file_path = os.path.join(root, file)
            
            try:
                # No header → header=None
                df = pd.read_csv(file_path, header=None)
                
                # First column = stages
                stages = df[0].dropna().unique()
                
                # Add to set
                unique_stages.update(stages)
                
            except Exception as e:
                print(f"Error in {file}: {e}")

# Convert to sorted list
unique_stages = sorted(unique_stages)

print("✅ Total Unique Stages:", len(unique_stages))
print("📌 Stages:")
for stage in unique_stages:
    print(stage)

✅ Total Unique Stages: 16
📌 Stages:
t2
t3
t4
t5
t6
t7
t8
t9+
tB
tEB
tHB
tM
tPB2
tPNa
tPNf
tSB


In [3]:
"""
setup_env.py  —  Cell 0. Must be the very first cell. No restart needed.
=========================================================================
Strategy
--------
Check the installed torch version using importlib.metadata (zero imports
of torch itself). If it's wrong, reinstall BEFORE torch is ever imported
into the process. This way Kaggle "Save and Run" works in one shot.

Required cell order in your notebook
-------------------------------------
  Cell 0 : exec(open("setup_env.py").read())   ← this file
  Cell 1 : from embryo_train import ...         ← first torch import
  Cell 2 : loaders = make_dataloaders(...)
  Cell 3 : train_all_models(loaders, ...)

DO NOT import torch before Cell 0 completes.
"""

import subprocess
import sys


# ── 1. Read installed versions WITHOUT importing torch ────────────────────────

def _installed_version(pkg: str) -> str:
    """Return installed version string, or '' if not found."""
    try:
        from importlib.metadata import version
        return version(pkg)
    except Exception:
        return ""


WANT_TORCH       = "2.5.1"
WANT_TORCHVISION = "0.20.1"
INDEX_URL        = "https://download.pytorch.org/whl/cu121"


def _version_ok() -> bool:
    tv = _installed_version("torch")
    tv2 = _installed_version("torchvision")
    return tv.startswith(WANT_TORCH) and tv2.startswith(WANT_TORCHVISION)


# ── 2. Install if wrong version (before any torch import) ─────────────────────

def ensure_compatible_torch():
    current = _installed_version("torch")

    if _version_ok():
        print(f"[OK] torch={_installed_version('torch')}  "
              f"torchvision={_installed_version('torchvision')}")
        print("[OK] Correct version already installed — no action needed.\n")
        return

    print(f"[INFO] Installed torch={current or 'not found'}  "
          f"(need {WANT_TORCH}+cu121 for P100 GPU)")
    print(f"[INFO] Installing torch=={WANT_TORCH} + "
          f"torchvision=={WANT_TORCHVISION} …")

    result = subprocess.run([
        sys.executable, "-m", "pip", "install",
        f"torch=={WANT_TORCH}",
        f"torchvision=={WANT_TORCHVISION}",
        "--index-url", INDEX_URL,
        "--force-reinstall",
        "--no-deps",
        "-q",
    ], capture_output=True, text=True)

    if result.returncode != 0:
        print("[ERROR] pip install failed:")
        print(result.stderr[-2000:])
        raise RuntimeError("Could not install compatible PyTorch. "
                           "Check your internet connection.")

    # Verify the files on disk changed
    if not _version_ok():
        print("[WARN] pip reported success but version still wrong.")
        print("       Try: Runtime > Restart and run again.")
    else:
        print(f"[OK] Installed torch={_installed_version('torch')}  "
              f"torchvision={_installed_version('torchvision')}")

    # Flush pip's cached dist-info so subsequent imports pick up new version
    import importlib
    import importlib.metadata
    try:
        importlib.invalidate_caches()
    except Exception:
        pass

    print("[OK] Install complete — no restart needed for Save & Run.\n")


# ── 3. Post-install GPU smoke test (only after install/confirm) ───────────────

def validate():
    """Run after ensure_compatible_torch(). First torch import happens here."""
    import torch
    import torch.nn as nn
    from torchvision import models

    print(f"[CHECK] torch={torch.__version__}  "
          f"torchvision={__import__('torchvision').__version__}")

    if not torch.cuda.is_available():
        print("[WARN] CUDA not available — will train on CPU (slow).")
        return

    # Strict GPU test: conv2d + hardtanh (the exact ops that fail on P100+cu128)
    try:
        x    = torch.randn(1, 3, 16, 16).cuda()
        conv = nn.Conv2d(3, 8, 3, padding=1).cuda()
        with torch.no_grad():
            _ = nn.Hardtanh()(conv(x))
        print(f"[OK] GPU conv2d test passed on {torch.cuda.get_device_name(0)}")
    except Exception as e:
        print(f"[ERROR] GPU conv2d test FAILED: {e}")
        print("        torch version on disk is correct but the OLD version")
        print("        is still loaded in memory.")
        print("        → This only happens when torch was imported before Cell 0.")
        print("        → Fix: Runtime > Restart Session, then re-run from Cell 0.")
        raise SystemExit("Please restart the kernel and re-run from Cell 0.")

    # Full model forward pass test
    print("[CHECK] Testing all model families on GPU …")
    for name, build, inp_size in [
        ("MobileNetV1 (custom)", None,                            224),
        ("MobileNetV2",  models.mobilenet_v2,                    224),
        ("VGG16-BN",     models.vgg16_bn,                        224),
        ("VGG19-BN",     models.vgg19_bn,                        224),
        ("InceptionV3",  models.inception_v3,                    299),
    ]:
        try:
            if build is None:
                # MobileNetV1 — test our custom implementation
                sys.path.insert(0, ".")
                from embryo_models import MobileNetV1
                m = MobileNetV1().cuda().eval()
            elif name == "InceptionV3":
                m = build(weights=None, aux_logits=False).cuda().eval()
            else:
                m = build(weights=None).cuda().eval()

            x_test = torch.randn(2, 3, inp_size, inp_size).cuda()
            with torch.no_grad():
                _ = m(x_test)
            print(f"  [OK] {name}")
            del m
            torch.cuda.empty_cache()
        except Exception as e:
            print(f"  [FAIL] {name}: {e}")

    print()
    print("=" * 50)
    print("  Environment summary")
    print("=" * 50)
    print(f"  torch       : {torch.__version__}")
    print(f"  torchvision : {__import__('torchvision').__version__}")
    print(f"  CUDA        : {torch.version.cuda}")
    print(f"  GPU         : {torch.cuda.get_device_name(0)}")
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"  VRAM        : {vram:.1f} GB")
    print("=" * 50)
    print("\n[READY] Safe to proceed.\n")


# ── Run ───────────────────────────────────────────────────────────────────────
ensure_compatible_torch()   # no torch import — safe to call unconditionally
validate()                  # first torch import happens here

[INFO] Installed torch=2.10.0+cu128  (need 2.5.1+cu121 for P100 GPU)
[INFO] Installing torch==2.5.1 + torchvision==0.20.1 …
[OK] Installed torch=2.5.1+cu121  torchvision=0.20.1+cu121
[OK] Install complete — no restart needed for Save & Run.

[CHECK] torch=2.5.1+cu121  torchvision=0.20.1+cu121
[OK] GPU conv2d test passed on Tesla P100-PCIE-16GB
[CHECK] Testing all model families on GPU …
  [FAIL] MobileNetV1 (custom): No module named 'embryo_models'
  [OK] MobileNetV2
  [OK] VGG16-BN
  [OK] VGG19-BN


/usr/local/lib/python3.12/dist-packages/torchvision/models/inception.py:43: FutureWarning: The default weight initialization of inception_v3 will be changed in future releases of torchvision. If you wish to keep the old behavior (which leads to long initialization times due to scipy/scipy#11299), please set init_weights=True.
  warnings.warn(


  [OK] InceptionV3

  Environment summary
  torch       : 2.5.1+cu121
  torchvision : 0.20.1+cu121
  CUDA        : 12.1
  GPU         : Tesla P100-PCIE-16GB
  VRAM        : 17.1 GB

[READY] Safe to proceed.



In [4]:
"""
embryo_dataset.py
=================
Dataset, DataLoader, transforms for embryo stage classification.

CSV format (no header, comma-separated):
    stage, start_frame, end_frame
    e.g.  tPNa,25,88

Frame filenames: any pattern where the RUN/frame number is the last
integer in the stem, e.g. D2016_S1_RUN42.jpeg  →  frame 42
"""

import os
import re
import csv
import math
import random
from pathlib import Path
from collections import Counter
from typing import Dict, List, Optional, Tuple

from PIL import Image, ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True   # tolerate corrupt/truncated jpegs

import torch
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms

# ── 16 stage labels ───────────────────────────────────────────────────────────
STAGES = [
    "t2", "t3", "t4", "t5", "t6", "t7", "t8", "t9+",
    "tB", "tEB", "tHB", "tM", "tPB2", "tPNa", "tPNf", "tSB",
]
LABEL2IDX: Dict[str, int] = {s: i for i, s in enumerate(STAGES)}
IDX2LABEL: Dict[int, str] = {i: s for i, s in enumerate(STAGES)}
NUM_CLASSES = len(STAGES)

# ── Transforms ────────────────────────────────────────────────────────────────
_MEAN = [0.485, 0.456, 0.406]
_STD  = [0.229, 0.224, 0.225]

TRAIN_TRANSFORM = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(_MEAN, _STD),
])

EVAL_TRANSFORM = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(_MEAN, _STD),
])


# ── Helpers ───────────────────────────────────────────────────────────────────

def _frame_number(filename: str) -> Optional[int]:
    nums = re.findall(r"\d+", Path(filename).stem)
    return int(nums[-1]) if nums else None


def _parse_csv(csv_path: Path) -> Dict[int, int]:
    frame_to_label: Dict[int, int] = {}
    with open(csv_path, newline="") as f:
        for row in csv.reader(f):
            if len(row) < 3:
                continue
            stage = row[0].strip()
            if stage not in LABEL2IDX:
                continue
            try:
                start, end = int(row[1]), int(row[2])
            except ValueError:
                continue
            label = LABEL2IDX[stage]
            for fn in range(start, end + 1):
                frame_to_label[fn] = label
    return frame_to_label


def build_sample_list(
    dataset_root: str,
    annotations_root: str,
) -> List[Tuple[Path, int, str]]:
    dataset_root     = Path(dataset_root)
    annotations_root = Path(annotations_root)
    samples: List[Tuple[Path, int, str]] = []
    skipped_csv = skipped_label = 0

    for video_dir in sorted(d for d in dataset_root.iterdir() if d.is_dir()):
        vid = video_dir.name
        csv_path = annotations_root / f"{vid}_phases.csv"
        if not csv_path.exists():
            skipped_csv += 1
            continue

        frame_to_label = _parse_csv(csv_path)
        frames = sorted(
            list(video_dir.glob("*.jpg")) + list(video_dir.glob("*.jpeg")),
            key=lambda p: _frame_number(p.name) or 0,
        )
        for img_path in frames:
            fn = _frame_number(img_path.name)
            if fn is None:
                continue
            label = frame_to_label.get(fn)
            if label is None:
                skipped_label += 1
                continue
            samples.append((img_path, label, vid))

    print(f"[INFO] Built {len(samples):,} samples across "
          f"{len(samples) - skipped_csv} videos.")
    if skipped_csv:
        print(f"       Skipped {skipped_csv} folder(s) — no CSV.")
    if skipped_label:
        print(f"       Skipped {skipped_label} frame(s) — outside annotated range.")
    return samples


def split_by_video(
    samples: List[Tuple[Path, int, str]],
    val_ratio: float = 0.15,
    test_ratio: float = 0.15,
    seed: int = 42,
) -> Tuple[List, List, List]:
    random.seed(seed)
    vids = sorted({s[2] for s in samples})
    random.shuffle(vids)
    n      = len(vids)
    n_val  = max(1, math.ceil(n * val_ratio))
    n_test = max(1, math.ceil(n * test_ratio))
    val_ids   = set(vids[:n_val])
    test_ids  = set(vids[n_val: n_val + n_test])
    train_ids = set(vids[n_val + n_test:])
    train = [s for s in samples if s[2] in train_ids]
    val   = [s for s in samples if s[2] in val_ids]
    test  = [s for s in samples if s[2] in test_ids]
    print(f"[INFO] Split → train: {len(train):,} ({len(train_ids)} videos) | "
          f"val: {len(val):,} ({len(val_ids)}) | "
          f"test: {len(test):,} ({len(test_ids)})")
    return train, val, test


# ── Dataset ───────────────────────────────────────────────────────────────────

class EmbryoDataset(Dataset):
    def __init__(self, samples: List[Tuple[Path, int, str]], transform=None):
        self.samples   = samples
        self.transform = transform

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx: int):
        # Retry neighbours if image is corrupt
        for attempt in range(len(self.samples)):
            try:
                img_path, label, _ = self.samples[(idx + attempt) % len(self.samples)]
                image = Image.open(img_path).convert("RGB")
                if self.transform:
                    image = self.transform(image)
                return image, label
            except Exception:
                continue
        # Absolute fallback
        return torch.zeros(3, 224, 224), self.samples[idx][1]

    def class_counts(self) -> Counter:
        return Counter(label for _, label, _ in self.samples)

    def class_weights(self) -> torch.Tensor:
        counts  = self.class_counts()
        weights = torch.zeros(NUM_CLASSES)
        for cls, cnt in counts.items():
            weights[cls] = 1.0 / cnt
        weights = weights / weights.sum() * NUM_CLASSES
        return weights

    def sample_weights(self) -> torch.Tensor:
        cw = self.class_weights()
        return torch.tensor([cw[label].item() for _, label, _ in self.samples])


# ── DataLoader factory ────────────────────────────────────────────────────────

def make_dataloaders(
    dataset_root: str,
    annotations_root: str,
    batch_size: int = 128,
    num_workers: int = 4,
    val_ratio: float = 0.15,
    test_ratio: float = 0.15,
    oversample_train: bool = True,
    seed: int = 42,
) -> Dict[str, DataLoader]:

    all_samples = build_sample_list(dataset_root, annotations_root)
    if not all_samples:
        raise RuntimeError("No samples found — check dataset_root and annotations_root.")

    train_s, val_s, test_s = split_by_video(
        all_samples, val_ratio=val_ratio, test_ratio=test_ratio, seed=seed
    )

    train_ds = EmbryoDataset(train_s, transform=TRAIN_TRANSFORM)
    val_ds   = EmbryoDataset(val_s,   transform=EVAL_TRANSFORM)
    test_ds  = EmbryoDataset(test_s,  transform=EVAL_TRANSFORM)

    print("\n[INFO] Train class distribution:")
    counts = train_ds.class_counts()
    for idx, name in IDX2LABEL.items():
        print(f"  {name:>6}: {counts.get(idx, 0):6,}")

    sampler       = None
    train_shuffle = True
    if oversample_train:
        sampler       = WeightedRandomSampler(train_ds.sample_weights(),
                                              len(train_ds), replacement=True)
        train_shuffle = False

    pin  = torch.cuda.is_available()
    pers = num_workers > 0

    loaders = {
        "train": DataLoader(train_ds, batch_size=batch_size,
                            sampler=sampler, shuffle=train_shuffle,
                            num_workers=num_workers, pin_memory=pin,
                            persistent_workers=pers),
        "val":   DataLoader(val_ds,   batch_size=batch_size, shuffle=False,
                            num_workers=num_workers, pin_memory=pin,
                            persistent_workers=pers),
        "test":  DataLoader(test_ds,  batch_size=batch_size, shuffle=False,
                            num_workers=num_workers, pin_memory=pin,
                            persistent_workers=pers),
    }
    print(f"\n[INFO] DataLoaders ready — "
          f"train: {len(loaders['train'])} batches | "
          f"val: {len(loaders['val'])} | "
          f"test: {len(loaders['test'])}\n")
    return loaders

In [5]:
# !pip install -q torchvision==0.20.1 --index-url https://download.pytorch.org/whl/cu121 --force-reinstall

In [6]:
# pd.read_csv('/kaggle/input/datasets/abhishekbuddiga06/embryo-dataset/embryo_dataset_annotations/embryo_dataset_annotations/AA83-7_phases.csv')
# video_dir = sorted(Path(DATASET_ROOT).iterdir())[0]
# video_id  = video_dir.name
csv_path  = '/kaggle/input/datasets/abhishekbuddiga06/embryo-dataset/embryo_dataset_annotations/embryo_dataset_annotations/AA83-7_phases.csv'

print(f"Checking: {csv_path}\n")

with open(csv_path, newline="") as f:
    raw = f.read()

print(repr(raw[:500]))   # show exact bytes including separators

Checking: /kaggle/input/datasets/abhishekbuddiga06/embryo-dataset/embryo_dataset_annotations/embryo_dataset_annotations/AA83-7_phases.csv

'tPB2,5,24\ntPNa,25,88\ntPNf,89,97\nt2,98,171\nt3,172,177\nt4,178,191\nt5,192,241\nt6,242,256\nt7,257,276\nt8,277,286\n'


In [7]:
DATASET_ROOT      = "/kaggle/input/datasets/abhishekbuddiga06/embryo-dataset/embryo_dataset/embryo_dataset"
ANNOTATIONS_ROOT  = "/kaggle/input/datasets/abhishekbuddiga06/embryo-dataset/embryo_dataset_annotations/embryo_dataset_annotations"
# ─────────────────────────────────────────────────────────────────────────

loaders = make_dataloaders(
    dataset_root=DATASET_ROOT,
    annotations_root=ANNOTATIONS_ROOT,
    batch_size=128,
    num_workers=4,          # set 0 on Windows if you hit multiprocessing issues
    oversample_train=True,
)

# peek at one batch
images, labels = next(iter(loaders["train"]))
print(f"Batch shape : {images.shape}")          # (32, 3, 224, 224)
print(f"Label tensor: {labels}")
print(f"Stage names : {[IDX2LABEL[l.item()] for l in labels[:8]]}")

[INFO] Built 297,428 samples across 297428 videos.
       Skipped 44935 frame(s) — outside annotated range.
[INFO] Split → train: 208,529 (492 videos) | val: 45,394 (106) | test: 43,505 (106)

[INFO] Train class distribution:
      t2: 20,258
      t3:  3,762
      t4: 20,149
      t5:  5,830
      t6:  5,624
      t7:  7,835
      t8: 22,947
     t9+: 36,051
      tB:  7,469
     tEB: 13,861
     tHB:     87
      tM: 12,267
    tPB2:  5,909
    tPNa: 29,878
    tPNf:  4,721
     tSB: 11,881

[INFO] DataLoaders ready — train: 1630 batches | val: 355 | test: 340

Batch shape : torch.Size([128, 3, 224, 224])
Label tensor: tensor([ 1,  9,  5, 14, 12, 14, 13,  4, 14, 10,  3,  7, 11,  2,  3,  6,  2, 12,
        15,  4, 10, 14, 15, 15, 14, 15,  3,  5, 14,  0, 15,  6, 10,  2, 10,  8,
        12,  0, 10,  7,  9,  2, 11,  0,  2,  7, 12,  9,  5, 14, 12,  7,  0,  7,
         5,  9,  8, 14,  0, 14, 10, 10, 10,  3, 13,  8, 12, 13,  3,  9,  2, 13,
         8,  5,  2,  1,  3,  4,  1,  9,  3,  6,  5,

In [8]:
from pathlib import Path

DATASET_ROOT     = "/kaggle/input/datasets/abhishekbuddiga06/embryo-dataset/embryo_dataset/embryo_dataset"
ANNOTATIONS_ROOT = "/kaggle/input/datasets/abhishekbuddiga06/embryo-dataset/embryo_dataset_annotations/embryo_dataset_annotations"

d = Path(DATASET_ROOT)
a = Path(ANNOTATIONS_ROOT)

print("=== DATASET ROOT ===")
print(f"Exists: {d.exists()} | Absolute: {d.resolve()}")
if d.exists():
    subdirs = list(d.iterdir())
    print(f"Contents ({len(subdirs)} items): {[x.name for x in subdirs[:5]]}")
    # Check first video folder
    first_video = next((x for x in subdirs if x.is_dir()), None)
    if first_video:
        frames = list(first_video.iterdir())
        print(f"\nFirst video '{first_video.name}' — {len(frames)} files")
        print(f"Sample filenames: {[f.name for f in frames[:5]]}")

print("\n=== ANNOTATIONS ROOT ===")
print(f"Exists: {a.exists()} | Absolute: {a.resolve()}")
if a.exists():
    csvs = list(a.iterdir())
    print(f"Contents ({len(csvs)} items): {[x.name for x in csvs[:5]]}")

print("\n=== CWD ===")
import os; print(os.getcwd())

=== DATASET ROOT ===
Exists: True | Absolute: /kaggle/input/datasets/abhishekbuddiga06/embryo-dataset/embryo_dataset/embryo_dataset
Contents (704 items): ['FM1017-5', 'HC459-6', 'GJ191-1', 'LMMG218-1-10', 'RD1142-2']

First video 'FM1017-5' — 454 files
Sample filenames: ['D2016.11.25_S1782_I132_WELL5_RUN18.jpeg', 'D2016.11.25_S1782_I132_WELL5_RUN62.jpeg', 'D2016.11.25_S1782_I132_WELL5_RUN378.jpeg', 'D2016.11.25_S1782_I132_WELL5_RUN58.jpeg', 'D2016.11.25_S1782_I132_WELL5_RUN263.jpeg']

=== ANNOTATIONS ROOT ===
Exists: True | Absolute: /kaggle/input/datasets/abhishekbuddiga06/embryo-dataset/embryo_dataset_annotations/embryo_dataset_annotations
Contents (704 items): ['BC750-7_phases.csv', 'SK308-7_phases.csv', 'QC697-4_phases.csv', 'DI358-3_phases.csv', 'DV210-4_phases.csv']

=== CWD ===
/kaggle/working


In [9]:
"""
embryo_models.py
================
Models  : MobileNetV1 (custom), MobileNetV2, InceptionV3, VGG16-BN, VGG19-BN
Loss    : HierarchicalEmbryoLoss

IMPORTANT: Import this file BEFORE torchvision so the legacy-pickle
           patch is applied in time for VGG weight loading.
"""

# ── Patch 1: missing legacy module (needed for VGG weights on PyTorch 2.5.x) -
import sys
import types as _types
for _mod in ("torch.utils.serialization",):
    if _mod not in sys.modules:
        sys.modules[_mod] = _types.ModuleType(_mod)

# ── Patch 2: force weights_only=False in torch.hub so VGG pickle loads ───────
import torch.hub as _hub
_orig_load = _hub.load_state_dict_from_url
def _patched_load(url, *args, **kwargs):
    kwargs["weights_only"] = False
    return _orig_load(url, *args, **kwargs)
_hub.load_state_dict_from_url = _patched_load
# ─────────────────────────────────────────────────────────────────────────────

import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import models
from typing import Dict, Optional

# from embryo_dataset import STAGES, NUM_CLASSES

# ── Developmental order & distance matrix ─────────────────────────────────────
ORDERED_STAGES = [
    "tPB2", "tPNa", "tPNf", "t2",  "t3",  "t4",  "t5",  "t6",
    "t7",   "t8",   "t9+",  "tM",  "tSB", "tB",  "tEB", "tHB",
]
_rank = {s: r for r, s in enumerate(ORDERED_STAGES)}
ORDINAL_RANK = torch.tensor([_rank[s] for s in STAGES], dtype=torch.float32)
DISTANCE_MATRIX = torch.abs(ORDINAL_RANK.unsqueeze(0) - ORDINAL_RANK.unsqueeze(1))


# ═════════════════════════════════════════════════════════════════════════════
# 1.  Custom Loss
# ═════════════════════════════════════════════════════════════════════════════

class HierarchicalEmbryoLoss(nn.Module):
    """
    L = alpha * CrossEntropy  +  (1-alpha) * HierarchicalPenalty

    where HierarchicalPenalty = Σ_i  softmax(logit_i) * |rank(i) - rank(true)|

    Desirable properties:
      ① Non-negativity      : CE ≥ 0, H ≥ 0  →  L ≥ 0
      ② Identity            : L = 0  iff  all probability mass on correct class
      ③ Differentiability   : softmax is C∞; distances are fixed constants
      ④ Ordinal sensitivity : penalty grows with developmental distance of error
      ⑤ Scale consistency   : H normalised by max distance → both terms ∈ [0,1]
      ⑥ Class-balance aware : optional per-class weights passed to CE
    """
    def __init__(
        self,
        alpha: float = 0.5,
        class_weights: Optional[torch.Tensor] = None,
        label_smoothing: float = 0.1,
        normalize_H: bool = True,
    ):
        super().__init__()
        assert 0.0 <= alpha <= 1.0
        self.alpha       = alpha
        self.normalize_H = normalize_H
        self._max_dist   = float(DISTANCE_MATRIX.max())
        self.ce = nn.CrossEntropyLoss(weight=class_weights,
                                      label_smoothing=label_smoothing)
        self.register_buffer("dist_matrix", DISTANCE_MATRIX)

    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        ce_loss        = self.ce(logits, targets)
        probs          = F.softmax(logits, dim=1)
        dist_to_target = self.dist_matrix[targets]
        h_loss         = (probs * dist_to_target).sum(dim=1).mean()
        if self.normalize_H:
            h_loss = h_loss / self._max_dist
        return self.alpha * ce_loss + (1.0 - self.alpha) * h_loss


# ═════════════════════════════════════════════════════════════════════════════
# 2.  MobileNetV1  (custom — not in torchvision)
# ═════════════════════════════════════════════════════════════════════════════

class _DWSBlock(nn.Sequential):
    """Depthwise-separable conv block."""
    def __init__(self, in_ch: int, out_ch: int, stride: int = 1):
        super().__init__(
            nn.Conv2d(in_ch, in_ch, 3, stride=stride, padding=1,
                      groups=in_ch, bias=False),
            nn.BatchNorm2d(in_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(in_ch, out_ch, 1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )

class MobileNetV1(nn.Module):
    """MobileNetV1 (Howard et al., 2017). Input: 224×224."""
    def __init__(self, num_classes: int = NUM_CLASSES, width_mult: float = 1.0):
        super().__init__()
        def c(n): return max(1, int(n * width_mult))
        self.features = nn.Sequential(
            nn.Conv2d(3, c(32), 3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(c(32)), nn.ReLU(inplace=True),
            _DWSBlock(c(32),   c(64),   1),
            _DWSBlock(c(64),   c(128),  2),
            _DWSBlock(c(128),  c(128),  1),
            _DWSBlock(c(128),  c(256),  2),
            _DWSBlock(c(256),  c(256),  1),
            _DWSBlock(c(256),  c(512),  2),
            *[_DWSBlock(c(512), c(512), 1) for _ in range(5)],
            _DWSBlock(c(512),  c(1024), 2),
            _DWSBlock(c(1024), c(1024), 1),
        )
        self.pool       = nn.AdaptiveAvgPool2d(1)
        self.classifier = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(c(1024), num_classes),
        )
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode="fan_out")
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.ones_(m.weight); nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, 0, 0.01); nn.init.zeros_(m.bias)

    def forward(self, x):
        return self.classifier(self.pool(self.features(x)).flatten(1))


# ═════════════════════════════════════════════════════════════════════════════
# 3.  torchvision models
# ═════════════════════════════════════════════════════════════════════════════

def build_mobilenet_v1(num_classes=NUM_CLASSES, pretrained=True,
                       freeze_backbone=False) -> nn.Module:
    print("  [INFO] MobileNetV1 has no torchvision pretrained weights — "
          "training from scratch.")
    return MobileNetV1(num_classes=num_classes)


def build_mobilenet_v2(num_classes=NUM_CLASSES, pretrained=True,
                       freeze_backbone=False) -> nn.Module:
    w     = models.MobileNet_V2_Weights.IMAGENET1K_V1 if pretrained else None
    model = models.mobilenet_v2(weights=w)
    if freeze_backbone:
        for p in model.parameters(): p.requires_grad = False
    in_f = model.classifier[1].in_features
    model.classifier = nn.Sequential(nn.Dropout(0.3), nn.Linear(in_f, num_classes))
    return model


def build_inception(num_classes=NUM_CLASSES, pretrained=True,
                    freeze_backbone=False) -> nn.Module:
    """InceptionV3 — resize to 299×299 handled in training loop."""
    w     = models.Inception_V3_Weights.IMAGENET1K_V1 if pretrained else None
    model = models.inception_v3(weights=w, aux_logits=True)
    if freeze_backbone:
        for p in model.parameters(): p.requires_grad = False
    model.fc           = nn.Sequential(nn.Dropout(0.4),
                                       nn.Linear(model.fc.in_features, num_classes))
    model.AuxLogits.fc = nn.Linear(model.AuxLogits.fc.in_features, num_classes)
    for p in model.fc.parameters():           p.requires_grad = True
    for p in model.AuxLogits.fc.parameters(): p.requires_grad = True
    return model


def build_vgg16(num_classes=NUM_CLASSES, pretrained=True,
                freeze_backbone=False) -> nn.Module:
    w     = models.VGG16_BN_Weights.IMAGENET1K_V1 if pretrained else None
    model = models.vgg16_bn(weights=w)
    if freeze_backbone:
        for p in model.parameters(): p.requires_grad = False
    model.classifier[6] = nn.Linear(model.classifier[6].in_features, num_classes)
    for p in model.classifier.parameters(): p.requires_grad = True
    return model


def build_vgg19(num_classes=NUM_CLASSES, pretrained=True,
                freeze_backbone=False) -> nn.Module:
    w     = models.VGG19_BN_Weights.IMAGENET1K_V1 if pretrained else None
    model = models.vgg19_bn(weights=w)
    if freeze_backbone:
        for p in model.parameters(): p.requires_grad = False
    model.classifier[6] = nn.Linear(model.classifier[6].in_features, num_classes)
    for p in model.classifier.parameters(): p.requires_grad = True
    return model


# ── Factory ───────────────────────────────────────────────────────────────────
_BUILDERS = {
    "mobilenet_v1": build_mobilenet_v1,
    "mobilenet_v2": build_mobilenet_v2,
    "inception":    build_inception,
    "vgg16":        build_vgg16,
    "vgg19":        build_vgg19,
}

def get_model(name: str, **kwargs) -> nn.Module:
    name = name.lower()
    if name not in _BUILDERS:
        raise ValueError(f"Unknown model '{name}'. Choose from: {list(_BUILDERS)}")
    return _BUILDERS[name](**kwargs)

def count_parameters(model: nn.Module) -> str:
    total     = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return f"Total: {total:,}  |  Trainable: {trainable:,}"

In [ ]:
"""
embryo_train.py
===============
Trains MobileNetV1, MobileNetV2, InceptionV3, VGG16, VGG19
with HierarchicalEmbryoLoss.

Features
--------
• Resumes automatically from the latest checkpoint if training is interrupted
• Saves checkpoint every N epochs AND whenever val_acc improves
• tqdm bars on both train and eval with live loss/acc
• Inception resized to 299×299 on GPU — no dataloader transform swap needed
• Single torch==2.5.1+cu121 / torchvision==0.20.1+cu121 requirement

Run
---
    comparison_df = train_all_models(loaders, num_epochs=15)
"""

# ── Must be first — patches VGG pickle loader before torchvision imports ──────
# import embryo_models  # noqa: F401  (side-effect: applies sys.modules patches)

import os, time, copy
from pathlib import Path
from typing import Dict, List, Tuple

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader
from tqdm import tqdm
import numpy as np
import pandas as pd
from sklearn.metrics import classification_report

# from embryo_dataset import (
#     STAGES, NUM_CLASSES, IDX2LABEL,
#     make_dataloaders, TRAIN_TRANSFORM, EVAL_TRANSFORM,
# )
# from embryo_models import (
#     HierarchicalEmbryoLoss, get_model,
#     ORDINAL_RANK, count_parameters,
# )

# ── Device ────────────────────────────────────────────────────────────────────
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[INFO] Using device: {DEVICE}")


# ═════════════════════════════════════════════════════════════════════════════
# 1.  Inception resize helper  (GPU-side, avoids dataloader transform issues)
# ═════════════════════════════════════════════════════════════════════════════

def _maybe_resize(images: torch.Tensor, model_name: str) -> torch.Tensor:
    """Resize to 299×299 on the GPU for Inception only."""
    if model_name == "inception" and images.shape[-1] != 299:
        images = F.interpolate(images, size=(299, 299),
                               mode="bilinear", align_corners=False)
    return images


# ═════════════════════════════════════════════════════════════════════════════
# 2.  Checkpoint helpers
# ═════════════════════════════════════════════════════════════════════════════

def _ckpt_path(save_dir: str, model_name: str, tag: str) -> str:
    return os.path.join(save_dir, f"{model_name}_{tag}.pt")


def save_checkpoint(path: str, model: nn.Module, optimizer, scheduler,
                    epoch: int, best_val_acc: float, history: dict):
    torch.save({
        "epoch":         epoch,
        "model_state":   model.state_dict(),
        "optimizer":     optimizer.state_dict(),
        "scheduler":     scheduler.state_dict(),
        "best_val_acc":  best_val_acc,
        "history":       history,
    }, path)


def load_checkpoint(path: str, model: nn.Module, optimizer, scheduler,
                    ) -> Tuple[int, float, dict]:
    """Load checkpoint into model/optimizer/scheduler in-place.
    Returns (start_epoch, best_val_acc, history).
    """
    ckpt = torch.load(path, map_location=DEVICE, weights_only=False)
    model.load_state_dict(ckpt["model_state"])
    optimizer.load_state_dict(ckpt["optimizer"])
    scheduler.load_state_dict(ckpt["scheduler"])
    print(f"  [RESUME] Loaded checkpoint '{path}'  "
          f"(epoch {ckpt['epoch']}, best_val={ckpt['best_val_acc']:.4f})")
    return ckpt["epoch"], ckpt["best_val_acc"], ckpt["history"]


# ═════════════════════════════════════════════════════════════════════════════
# 3.  Training — one epoch
# ═════════════════════════════════════════════════════════════════════════════

def train_one_epoch(
    model: nn.Module,
    loader: DataLoader,
    criterion: nn.Module,
    optimizer: optim.Optimizer,
    scheduler,
    model_name: str,
    epoch: int,
    num_epochs: int,
    scaler,
) -> Tuple[float, float]:
    model.train()
    total_loss = correct = total = 0

    desc = f"  [Epoch {epoch:02d}/{num_epochs}] Train"
    # pbar = tqdm(loader, desc=desc, leave=True, ncols=110, colour="green")

    for images, labels in loader:
        images = images.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)
        images = _maybe_resize(images, model_name)

        optimizer.zero_grad()
        with torch.amp.autocast("cuda"):
            if model_name == "inception":
                outputs, aux = model(images)
                loss = criterion(outputs, labels) + 0.4 * criterion(aux, labels)
            else:
                outputs = model(images)
                loss    = criterion(outputs, labels)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()

        bs          = images.size(0)
        total_loss += loss.item() * bs
        correct    += (outputs.argmax(1) == labels).sum().item()
        total      += bs
        # pbar.set_postfix(loss=f"{total_loss/total:.4f}",
        #                  acc=f"{correct/total:.4f}")

    scheduler.step()
    return total_loss / total, correct / total


# ═════════════════════════════════════════════════════════════════════════════
# 4.  Evaluation
# ═════════════════════════════════════════════════════════════════════════════

@torch.inference_mode()
def evaluate(
    model: nn.Module,
    loader: DataLoader,
    criterion: nn.Module,
    model_name: str,
    split: str = "Val",
) -> Tuple[float, float, List, List]:
    model.eval()
    total_loss = correct = total = 0
    all_preds, all_targets = [], []

    # desc = f"  [{split:>4}]"
    # pbar = tqdm(loader, desc=desc, leave=True, ncols=110, colour="blue")

    for images, labels in loader:
        images = images.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)
        images = _maybe_resize(images, model_name)

        outputs = model(images)
        if isinstance(outputs, tuple):
            outputs = outputs[0]

        loss  = criterion(outputs, labels)
        preds = outputs.argmax(1)

        bs          = images.size(0)
        total_loss += loss.item() * bs
        correct    += (preds == labels).sum().item()
        total      += bs
        all_preds.extend(preds.cpu().tolist())
        all_targets.extend(labels.cpu().tolist())
        # pbar.set_postfix(loss=f"{total_loss/total:.4f}",
        #                  acc=f"{correct/total:.4f}")

    return total_loss / total, correct / total, all_preds, all_targets


def mean_ordinal_error(preds: List[int], targets: List[int]) -> float:
    ranks  = ORDINAL_RANK.numpy()
    return float(np.mean([abs(ranks[p] - ranks[t]) for p, t in zip(preds, targets)]))


# ═════════════════════════════════════════════════════════════════════════════
# 5.  Full training loop — one model
# ═════════════════════════════════════════════════════════════════════════════

def train_model(
    model_name: str,
    loaders: Dict[str, DataLoader],
    num_epochs: int = 15,
    lr: float = 1e-4,
    alpha: float = 0.5,
    weight_decay: float = 1e-4,
    save_dir: str = "/kaggle/working/checkpoints",
    checkpoint_every: int = 5,
) -> Dict:
    print(f"\n{'='*60}")
    print(f"  Training: {model_name.upper()}")
    print(f"{'='*60}")
    os.makedirs(save_dir, exist_ok=True)

    # ── Build model, loss, optimiser, scheduler, scaler ───────────────────────
    model = get_model(model_name, pretrained=True, freeze_backbone=False).to(DEVICE)
    print(f"Parameters — {count_parameters(model)}\n")

    class_weights = loaders["train"].dataset.class_weights().to(DEVICE)
    criterion = HierarchicalEmbryoLoss(
        alpha=alpha, class_weights=class_weights,
        label_smoothing=0.1, normalize_H=True,
    ).to(DEVICE)

    optimizer = optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=lr, weight_decay=weight_decay,
    )
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs)
    scaler    = torch.amp.GradScaler("cuda") if DEVICE.type == "cuda" else \
                torch.amp.GradScaler("cpu")

    # ── Resume from latest checkpoint if available ────────────────────────────
    start_epoch  = 1
    best_val_acc = 0.0
    best_weights = copy.deepcopy(model.state_dict())
    history      = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}

    # Look for most recent periodic checkpoint first, then best checkpoint
    resume_path = None
    latest_epoch = 0
    for ep in range(num_epochs, 0, -checkpoint_every):
        candidate = _ckpt_path(save_dir, model_name, f"epoch{ep:03d}")
        if os.path.exists(candidate) and ep > latest_epoch:
            latest_epoch = ep
            resume_path  = candidate
            break

    if resume_path and latest_epoch < num_epochs:
        start_epoch, best_val_acc, history = load_checkpoint(
            resume_path, model, optimizer, scheduler
        )
        start_epoch += 1   # continue from NEXT epoch
        best_weights = copy.deepcopy(model.state_dict())
    elif os.path.exists(_ckpt_path(save_dir, model_name, "best")):
        # No periodic checkpoint but best exists — unlikely to need resume
        # but load best weights silently so test eval uses best
        ckpt = torch.load(_ckpt_path(save_dir, model_name, "best"),
                          map_location=DEVICE, weights_only=False)
        best_val_acc = ckpt["best_val_acc"]
        best_weights = ckpt["model_state"]
        print(f"  [INFO] Found best checkpoint (val={best_val_acc:.4f}) — "
              f"will update if improved.")

    if start_epoch > num_epochs:
        print(f"  [INFO] Already completed {num_epochs} epochs — skipping training.")
    else:
        # ── Training epochs ───────────────────────────────────────────────────
        for epoch in range(start_epoch, num_epochs + 1):
            t0 = time.time()

            train_loss, train_acc = train_one_epoch(
                model, loaders["train"], criterion, optimizer,
                scheduler, model_name, epoch, num_epochs, scaler,
            )
            val_loss, val_acc, _, _ = evaluate(
                model, loaders["val"], criterion, model_name, split="Val"
            )
            elapsed = time.time() - t0

            history["train_loss"].append(train_loss)
            history["train_acc"].append(train_acc)
            history["val_loss"].append(val_loss)
            history["val_acc"].append(val_acc)

            print(
                f"  ↳ Epoch {epoch:02d}/{num_epochs} | "
                f"train loss {train_loss:.4f}  acc {train_acc:.4f} | "
                f"val loss {val_loss:.4f}  acc {val_acc:.4f} | "
                f"{elapsed:.0f}s"
            )

            # ── Best checkpoint ───────────────────────────────────────────────
            if val_acc > best_val_acc:
                best_val_acc = val_acc
                best_weights = copy.deepcopy(model.state_dict())
                best_path    = _ckpt_path(save_dir, model_name, "best")
                save_checkpoint(best_path, model, optimizer, scheduler,
                                epoch, best_val_acc, history)
                print(f" Best checkpoint → {best_path}  "
                      f"(val_acc={best_val_acc:.4f})")

            # ── Periodic checkpoint every N epochs ────────────────────────────
            if epoch % checkpoint_every == 0:
                periodic_path = _ckpt_path(save_dir, model_name, f"epoch{epoch:03d}")
                save_checkpoint(periodic_path, model, optimizer, scheduler,
                                epoch, best_val_acc, history)
                print(f" Periodic checkpoint → {periodic_path}")

            print()  # blank line between epochs

    # ── Test evaluation ───────────────────────────────────────────────────────
    print(f"\n{'─'*60}")
    print("  Running TEST evaluation with best weights …")
    model.load_state_dict(best_weights)
    test_loss, test_acc, preds, targets = evaluate(
        model, loaders["test"], criterion, model_name, split="Test"
    )
    maoe = mean_ordinal_error(preds, targets)

    print(f"\n  [TEST] loss={test_loss:.4f}  acc={test_acc:.4f}  MAOE={maoe:.3f}\n")
    print(classification_report(targets, preds, target_names=STAGES, zero_division=0))

    return {
        "model":     model_name,
        "test_acc":  round(test_acc,  4),
        "test_loss": round(test_loss, 4),
        "maoe":      round(maoe,      3),
        "best_val":  round(best_val_acc, 4),
        "history":   history,
        "preds":     preds,
        "targets":   targets,
    }


# ═════════════════════════════════════════════════════════════════════════════
# 6.  Train all models & compare
# ═════════════════════════════════════════════════════════════════════════════

def train_all_models(
    loaders: Dict[str, DataLoader],
    vgg_loaders: Dict[str, DataLoader],
    num_epochs: int = 15,
    alpha: float = 0.5,
    checkpoint_every: int = 5,
    save_dir: str = "/kaggle/working/checkpoints",
) -> pd.DataFrame:
    model_names = ["mobilenet_v1", "mobilenet_v2", "inception", "vgg16", "vgg19"]
    results = []

    for name in model_names:
        if name == "vgg16" or name == "vgg19":
            r = train_model(name, vgg_loaders, num_epochs=num_epochs, alpha=alpha,
                            checkpoint_every=checkpoint_every, save_dir=save_dir)
        else:
            r = train_model(name, loaders, num_epochs=num_epochs, alpha=alpha,
                            checkpoint_every=checkpoint_every, save_dir=save_dir)
        results.append({
            "Model":        r["model"],
            "Test Acc":     r["test_acc"],
            "Test Loss":    r["test_loss"],
            "MAOE":         r["maoe"],
            "Best Val Acc": r["best_val"],
        })

    df = pd.DataFrame(results).sort_values("Test Acc", ascending=False)
    print("\n" + "="*55)
    print("  FINAL COMPARISON")
    print("="*55)
    print(df.to_string(index=False))
    out_csv = os.path.join(save_dir, "model_comparison.csv")
    df.to_csv(out_csv, index=False)
    print(f"\nSaved → {out_csv}")
    return df


# ═════════════════════════════════════════════════════════════════════════════
# 7.  Entry point
# ═════════════════════════════════════════════════════════════════════════════

if __name__ == "__main__":
    DATASET_ROOT     = "/kaggle/input/datasets/abhishekbuddiga06/embryo-dataset/embryo_dataset/embryo_dataset"
    ANNOTATIONS_ROOT = "/kaggle/input/datasets/abhishekbuddiga06/embryo-dataset/embryo_dataset_annotations/embryo_dataset_annotations"

    loaders = make_dataloaders(
        dataset_root=DATASET_ROOT,
        annotations_root=ANNOTATIONS_ROOT,
        batch_size=128,
        num_workers=4,
        oversample_train=True,
    )
    vgg_loaders = make_dataloaders(
        dataset_root=DATASET_ROOT,
        annotations_root=ANNOTATIONS_ROOT,
        batch_size=64,
        num_workers=4,
        oversample_train=True,
    )

    comparison_df = train_all_models(loaders, vgg_loaders, num_epochs=5, alpha=0.5,
                                     checkpoint_every=2)

[INFO] Using device: cuda
[INFO] Built 297,428 samples across 297428 videos.
       Skipped 44935 frame(s) — outside annotated range.
[INFO] Split → train: 208,529 (492 videos) | val: 45,394 (106) | test: 43,505 (106)

[INFO] Train class distribution:
      t2: 20,258
      t3:  3,762
      t4: 20,149
      t5:  5,830
      t6:  5,624
      t7:  7,835
      t8: 22,947
     t9+: 36,051
      tB:  7,469
     tEB: 13,861
     tHB:     87
      tM: 12,267
    tPB2:  5,909
    tPNa: 29,878
    tPNf:  4,721
     tSB: 11,881

[INFO] DataLoaders ready — train: 1630 batches | val: 355 | test: 340

[INFO] Built 297,428 samples across 297428 videos.
       Skipped 44935 frame(s) — outside annotated range.
[INFO] Split → train: 208,529 (492 videos) | val: 45,394 (106) | test: 43,505 (106)

[INFO] Train class distribution:
      t2: 20,258
      t3:  3,762
      t4: 20,149
      t5:  5,830
      t6:  5,624
      t7:  7,835
      t8: 22,947
     t9+: 36,051
      tB:  7,469
     tEB: 13,861
     tHB

  [Epoch 01/5] Train:   2%|▍                       | 28/1630 [00:14<09:16,  2.88it/s, acc=0.0633, loss=1.1521]